# 🧠 YOLOv11m Fine-Tuning — Purplle Retail Detection

This notebook trains a **YOLOv11m** person detector fine-tuned on frames extracted from your Purplle store CCTV clips, then exports to **ONNX** for CPU inference.

## Prerequisites
1. **Runtime → Change runtime type → GPU (T4 or A100)**
2. Upload your CCTV video clips to Google Drive OR mount Drive and reference them directly
3. Run each cell in order — they are numbered and self-contained

## What gets produced
- `yolov11m_retail.onnx` — the model file to drop into `models/`
- `runs/detect/train/` — training metrics and weight checkpoints

In [ ]:
# ============================================================
# CELL 1 — Verify GPU is available
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU detected — switch runtime to GPU')

In [ ]:
# ============================================================
# CELL 2 — Install dependencies
# ============================================================
!pip install -q ultralytics>=8.3.0 opencv-python-headless roboflow
import ultralytics
ultralytics.checks()
print(f'ultralytics version: {ultralytics.__version__}')

In [ ]:
# ============================================================
# CELL 2b - Download YOLOv11m pretrained weights
#
# ultralytics sometimes fails to auto-download in Colab.
# This cell fetches yolo11m.pt directly from the Ultralytics
# GitHub releases so the next cells can load it reliably.
# ============================================================
import os, subprocess

MODEL_FILE = 'yolo11m.pt'
DOWNLOAD_URL = 'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11m.pt'

if not os.path.exists(MODEL_FILE):
    print(f'Downloading {MODEL_FILE} from Ultralytics releases...')
    result = subprocess.run(
        ['wget', '-q', '--show-progress', '-O', MODEL_FILE, DOWNLOAD_URL],
        capture_output=False
    )
    if result.returncode != 0:
        raise RuntimeError(f'Download failed. Try manually: wget {DOWNLOAD_URL}')
    print(f'Downloaded: {os.path.getsize(MODEL_FILE)/1e6:.1f} MB')
else:
    print(f'Already present: {MODEL_FILE} ({os.path.getsize(MODEL_FILE)/1e6:.1f} MB)')

print('yolo11m.pt ready.')

In [ ]:
# ============================================================
# PyTorch 2.6 compatibility fix
#
# PyTorch 2.6 changed torch.load() default weights_only=True
# which blocks ultralytics .pt files. This cell patches the
# safe globals list so ultralytics models load correctly.
# Safe to run — yolo11m.pt is from the official Ultralytics repo.
# ============================================================
import torch

# Patch for PyTorch >= 2.6
if hasattr(torch.serialization, 'add_safe_globals'):
    try:
        from ultralytics.nn.tasks import DetectionModel
        from ultralytics.nn.modules import (
            Conv, C2f, SPPF, Detect, DFL
        )
        torch.serialization.add_safe_globals([
            DetectionModel,
            Conv, C2f, SPPF, Detect, DFL,
        ])
        print('Torch safe globals patched for ultralytics.')
    except ImportError:
        # Ultralytics not yet installed — will be handled at load time
        print('Ultralytics not yet installed; safe globals will be set after install.')
else:
    print(f'PyTorch {torch.__version__} — no safe_globals patch needed.')


In [ ]:
# ============================================================
# CELL 3 - Mount Drive and verify yolo_dataset from preprocess step
#
# PREREQUISITE: Run 00_preprocess_data.ipynb first.
# That notebook extracted frames, pseudo-labelled them, built the
# 80/20 train/val split, and saved everything to Drive already.
# This notebook only needs the ready-made yolo_dataset/ folder.
# You do NOT need the raw .mp4 clips for this notebook.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Change DRIVE_BASE only if you used a different folder in the preprocess notebook
DRIVE_BASE       = '/content/drive/MyDrive/purplle'
YOLO_DATASET_DIR = os.path.join(DRIVE_BASE, 'yolo_dataset')

assert os.path.exists(YOLO_DATASET_DIR), (
    f'yolo_dataset not found at: {YOLO_DATASET_DIR}\n'
    'Ensure 00_preprocess_data.ipynb ran successfully and used the same DRIVE_BASE.'
)

train_dir = os.path.join(YOLO_DATASET_DIR, 'images', 'train')
val_dir   = os.path.join(YOLO_DATASET_DIR, 'images', 'val')
n_train   = len(os.listdir(train_dir)) if os.path.exists(train_dir) else 0
n_val     = len(os.listdir(val_dir))   if os.path.exists(val_dir)   else 0

print(f'Found yolo_dataset at: {YOLO_DATASET_DIR}')
print(f'  Train images : {n_train}')
print(f'  Val images   : {n_val}')
print()
print('Frame extraction already completed by 00_preprocess_data.ipynb. Ready to train.')

In [ ]:
# ============================================================
# CELL 4 - Copy yolo_dataset from Drive to local /content/dataset
#
# Training directly from Drive is ~3-5x slower than local SSD.
# This copy takes 2-5 min but makes training much faster.
# ============================================================
import shutil, time

LOCAL_DATASET = '/content/dataset'

if os.path.exists(LOCAL_DATASET):
    shutil.rmtree(LOCAL_DATASET)

print(f'Copying {YOLO_DATASET_DIR} -> {LOCAL_DATASET} ...')
t0 = time.time()
shutil.copytree(YOLO_DATASET_DIR, LOCAL_DATASET)
elapsed = time.time() - t0

n_train = len(os.listdir(os.path.join(LOCAL_DATASET, 'images', 'train')))
n_val   = len(os.listdir(os.path.join(LOCAL_DATASET, 'images', 'val')))
print(f'Done in {elapsed:.0f}s | Train: {n_train} images | Val: {n_val} images')

In [ ]:
# ============================================================
# CELL 5 - Write dataset YAML (points to local /content/dataset)
# ============================================================
import yaml

dataset_yaml = {
    'path':  '/content/dataset',
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': ['person'],
}

with open('/content/retail.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f)

print('Dataset YAML written:')
print(open('/content/retail.yaml').read())

In [ ]:
# ============================================================
# CELL 8 — Fine-tune YOLOv11m on retail frames
#
# Training params follow design.md §8.1:
#   epochs=50, imgsz=640, batch=8, conf=0.3
#
# Expected training time on T4:
#   ~500 frames  → ~10 min
#   ~2000 frames → ~40 min
#   ~5000 frames → ~90 min
# ============================================================
from ultralytics import YOLO

model = YOLO('yolo11m.pt')   # Start from pretrained weights

results = model.train(
    data='/content/retail.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,          # GPU
    conf=0.3,          # Low threshold — ByteTrack handles FP filtering
    iou=0.45,          # Soft-NMS equivalent
    cos_lr=True,       # Cosine annealing
    augment=True,      # Mosaic, color jitter, flip augmentation
    project='/content/runs/detect',
    name='yolov11m_retail',
    save=True,
    patience=15,       # Early stopping
    cache=True,        # Cache images in RAM for faster training
)

print('Training complete!')
print(f'Best mAP@0.5: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')

In [ ]:
# ============================================================
# CELL 9 — Validate on held-out frames
# Target: mAP@0.5 > 0.70 for retail crowd detection
# ============================================================
from ultralytics import YOLO

best_model_path = '/content/runs/detect/yolov11m_retail/weights/best.pt'
model = YOLO(best_model_path)

metrics = model.val(data='/content/retail.yaml', conf=0.3, iou=0.45)

map50   = metrics.box.map50
map5095 = metrics.box.map

print(f'mAP@0.5:      {map50:.4f}  (target: > 0.70)')
print(f'mAP@0.5:0.95: {map5095:.4f}')

if map50 < 0.70:
    print('⚠️  mAP below target. Consider: more epochs, more diverse frames, or manual annotation review.')
else:
    print('✅  Model meets target accuracy. Proceeding to ONNX export.')

In [ ]:
# ============================================================
# CELL 10 — Export to ONNX
# Flags match design.md §8.1: dynamic=True, simplify=True
# ============================================================
from ultralytics import YOLO

best_model_path = '/content/runs/detect/yolov11m_retail/weights/best.pt'
model = YOLO(best_model_path)

onnx_path = model.export(
    format='onnx',
    dynamic=True,       # Variable batch size
    simplify=True,      # Graph optimisation via onnx-simplifier
    opset=17,
    imgsz=640,
)

print(f'ONNX model exported to: {onnx_path}')

# Verify ONNX model integrity
import onnx
m = onnx.load(onnx_path)
onnx.checker.check_model(m)
print('✅  ONNX model validation passed')

In [ ]:
# ============================================================
# CELL 11 — Benchmark ONNX inference speed on CPU
# Target: ≥ 10 fps per camera (design.md §Requirement 2.6)
# ============================================================
import onnxruntime as ort
import numpy as np
import time

onnx_path = '/content/runs/detect/yolov11m_retail/weights/best.onnx'
session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

# Warmup
dummy = np.random.rand(1, 3, 640, 640).astype(np.float32)
for _ in range(5):
    session.run(None, {session.get_inputs()[0].name: dummy})

# Benchmark 50 iterations
N = 50
t0 = time.perf_counter()
for _ in range(N):
    session.run(None, {session.get_inputs()[0].name: dummy})
elapsed = time.perf_counter() - t0

avg_ms = (elapsed / N) * 1000
fps = 1000 / avg_ms

print(f'Average inference time: {avg_ms:.1f} ms')
print(f'Throughput: {fps:.1f} fps')

if fps >= 10:
    print('✅  Meets ≥10 fps requirement on CPU')
else:
    print(f'⚠️  Below target. Consider quantisation (INT8) or reducing imgsz to 480.')

In [ ]:
# ============================================================
# CELL 12 — Copy ONNX model to Google Drive for download
# Then download it and place in your local models/ directory
# ============================================================
import shutil, os

onnx_src = '/content/runs/detect/yolov11m_retail/weights/best.onnx'
drive_dest = '/content/drive/MyDrive/purplle/models/yolov11m_retail.onnx'
os.makedirs(os.path.dirname(drive_dest), exist_ok=True)
shutil.copy(onnx_src, drive_dest)

print(f'✅  Model saved to Google Drive: {drive_dest}')
print()
print('Next steps:')
print('  1. Download yolov11m_retail.onnx from Google Drive')
print('  2. Place it in your local: models/yolov11m_retail.onnx')
print('  3. Restart the vision-pipeline container:')
print('     docker compose restart vision-pipeline')
print('  The pipeline will auto-detect and load the ONNX model on startup.')

# Also download directly from Colab if preferred
from google.colab import files
files.download(onnx_src)

## 📊 Training Summary

After running all cells:

| Item | Value |
|------|-------|
| Base model | YOLOv11m (pretrained COCO) |
| Dataset | Frames extracted from your 8 CCTV clips |
| Export format | ONNX (dynamic batch, simplified graph) |
| Target mAP@0.5 | > 0.70 |
| Target CPU speed | ≥ 10 fps |
| Output file | `yolov11m_retail.onnx` → place in `models/` |

### If mAP < 0.70:
- Increase epochs to 100
- Manually review/fix pseudo-labels for 50–100 difficult frames (dense crowds, heavy occlusion)
- Add frames from different lighting conditions (morning vs evening lighting)
- Reduce `FRAME_SAMPLE_INTERVAL` in Cell 4 to extract more frames

### If speed < 10 fps:
```python
# INT8 quantisation (run after Cell 10):
!pip install -q onnxconverter-common
from onnxconverter_common import float16
import onnx
model_fp32 = onnx.load('best.onnx')
model_fp16 = float16.convert_float_to_float16(model_fp32)
onnx.save(model_fp16, 'best_fp16.onnx')
```